# CUDA tutorial: CPU vs GPU benchmarks

This notebook runs four classic problems twice, **sequentially on one CPU core** and **in parallel on the GPU**, checks every GPU answer against the CPU, and charts the timings:

1. **Vector addition**: `c[i] = a[i] + b[i]`
2. **Matrix multiplication**: CPU loop, naive GPU kernel, tiled shared-memory kernel, cuBLAS
3. **Image blur**: box filter, one GPU thread per pixel
4. **Summation**: a race condition, atomics, and a shared-memory tree reduction

**Before you start**

- **Google Colab:** Runtime → Change runtime type → **T4 GPU** → Save.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (the plain "GPU" option is an older P100).
- Then **Run all**. The whole notebook takes about 2–3 minutes.

Everything it measures is saved in `cuda_examples/results/` (CSV files and PNG charts), and the last cell bundles them into `cuda_results.zip`.

## 1. Check the GPU and the CUDA compiler

In [ ]:
!nvidia-smi
!nvcc --version
!lscpu | grep -E "Model name|^CPU\(s\)"

: 

In [ ]:
import os, shutil, subprocess

# Work inside cuda_examples/ (safe to re-run this cell).
if os.path.basename(os.getcwd()) != "cuda_examples":
    os.makedirs("cuda_examples", exist_ok=True)
    os.chdir("cuda_examples")
os.makedirs("results", exist_ok=True)

if shutil.which("nvcc") is None:
    raise SystemExit("nvcc not found. Use a Colab or Kaggle GPU runtime, or install the CUDA Toolkit.")

query = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                       capture_output=True, text=True)
if query.returncode != 0 or not query.stdout.strip():
    raise SystemExit("No GPU found. Switch the runtime to a GPU (see the top of the notebook) and run again.")

gpu_name, cap = [s.strip() for s in query.stdout.strip().splitlines()[0].split(",")]
ARCH = "sm_" + cap.replace(".", "")
print(f"GPU: {gpu_name}, compute capability {cap} -> compiling with -arch={ARCH}")

## 2. Write the source files

These are the same files as in `cuda_examples.zip`. `common.h` holds the timers and the CSV output; each `.cu` file has a CPU function, one or more GPU kernels, and a `run()` that times both and checks the GPU result.

In [ ]:
%%writefile common.h
// common.h - small helpers shared by the four CUDA tutorial examples.
//
// Every example prints one CSV row per measurement:
//   example,variant,size,time_ms,throughput,unit,error,check
// Lines that start with '#' are comments (for example, which GPU was used).
#pragma once

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <random>
#include <string>
#include <vector>

#include <cuda_runtime.h>

// Abort with a readable message if a CUDA API call fails.
#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err_ = (call);                                             \
        if (err_ != cudaSuccess) {                                             \
            std::fprintf(stderr, "CUDA error at %s:%d: %s\n  in: %s\n",        \
                         __FILE__, __LINE__, cudaGetErrorString(err_), #call); \
            std::exit(EXIT_FAILURE);                                           \
        }                                                                      \
    } while (0)

// A kernel launch returns no error code, so ask for the last error right after it.
#define CUDA_CHECK_LAUNCH() CUDA_CHECK(cudaGetLastError())

// Wall-clock timer for CPU code.
struct CpuTimer {
    std::chrono::steady_clock::time_point t0;
    void start() { t0 = std::chrono::steady_clock::now(); }
    double stop_ms() const {
        return std::chrono::duration<double, std::milli>(
                   std::chrono::steady_clock::now() - t0).count();
    }
};

// GPU timer built on CUDA events. Kernel launches return immediately, so a CPU
// timer would only measure the launch. Events are timestamped by the GPU itself.
struct GpuTimer {
    cudaEvent_t begin{}, end{};
    GpuTimer() {
        CUDA_CHECK(cudaEventCreate(&begin));
        CUDA_CHECK(cudaEventCreate(&end));
    }
    ~GpuTimer() {
        cudaEventDestroy(begin);
        cudaEventDestroy(end);
    }
    void start() { CUDA_CHECK(cudaEventRecord(begin)); }
    double stop_ms() {
        CUDA_CHECK(cudaEventRecord(end));
        CUDA_CHECK(cudaEventSynchronize(end));  // wait until the GPU reaches 'end'
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, begin, end));
        return ms;
    }
};

// Call fn() (which returns elapsed milliseconds) once as a warm-up, then `reps`
// more times, and return the median. A first run slower than `slow_ms` is
// returned as is, so multi-second CPU runs are not repeated.
template <class F>
double median_ms(F fn, int reps = 5, double slow_ms = 2000.0) {
    const double first = fn();
    if (first > slow_ms) return first;
    std::vector<double> t;
    for (int i = 0; i < reps; ++i) t.push_back(fn());
    std::sort(t.begin(), t.end());
    return t[t.size() / 2];
}

// Fill a vector with reproducible random numbers in [lo, hi).
inline void fill_random(std::vector<float>& v, float lo, float hi, unsigned seed) {
    std::mt19937 rng(seed);
    std::uniform_real_distribution<float> dist(lo, hi);
    for (float& x : v) x = dist(rng);
}

inline void print_csv_header() {
    std::printf("example,variant,size,time_ms,throughput,unit,error,check\n");
}

inline void print_row(const char* example, const char* variant, long long size,
                      double ms, double throughput, const char* unit,
                      double error, const char* check) {
    std::printf("%s,%s,%lld,%.4f,%.3f,%s,%.3e,%s\n", example, variant, size, ms,
                throughput, unit, error, check);
    std::fflush(stdout);
}

inline void print_device_info() {
    int dev = 0;
    cudaDeviceProp p{};
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaGetDeviceProperties(&p, dev));
    std::printf("# GPU: %s | compute capability %d.%d | %d SMs | %.1f GB\n",
                p.name, p.major, p.minor, p.multiProcessorCount,
                p.totalGlobalMem / 1e9);
}

// --- tiny command-line helpers ---------------------------------------------

inline bool has_flag(int argc, char** argv, const char* flag) {
    for (int i = 1; i < argc; ++i)
        if (std::strcmp(argv[i], flag) == 0) return true;
    return false;
}

// Value after an option such as "--radius 4"; `fallback` if it is absent.
inline const char* option_value(int argc, char** argv, const char* opt,
                                const char* fallback) {
    for (int i = 1; i + 1 < argc; ++i)
        if (std::strcmp(argv[i], opt) == 0) return argv[i + 1];
    return fallback;
}

// Positional numbers (problem sizes). Options that take a value are skipped,
// along with their value; "--cpu-only" takes no value.
inline std::vector<long long> sizes_from_args(int argc, char** argv,
                                              std::vector<long long> defaults) {
    std::vector<long long> sizes;
    for (int i = 1; i < argc; ++i) {
        if (argv[i][0] == '-' && argv[i][1] == '-') {
            if (std::strcmp(argv[i], "--cpu-only") != 0) ++i;  // skip its value
            continue;
        }
        sizes.push_back(std::atoll(argv[i]));
    }
    return sizes.empty() ? defaults : sizes;
}

In [ ]:
%%writefile vector_add.cu
// vector_add.cu - c[i] = a[i] + b[i]
//
// CPU: one core walks the array from start to end.
// GPU: one thread per element; all of them run at once.
//
// Build: nvcc -O3 -std=c++17 -arch=native vector_add.cu -o vector_add
// Run:   ./vector_add                 (default sizes)
//        ./vector_add 1000000 50000000 (your own sizes)
#include "common.h"

// Sequential version: a plain loop.
void vector_add_cpu(const float* a, const float* b, float* c, int n) {
    for (int i = 0; i < n; ++i) c[i] = a[i] + b[i];
}

// Parallel version: the loop is gone. Each thread works out which element it
// owns from its block number and its position inside the block.
__global__ void vector_add_gpu(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // this thread's global index
    if (i < n) c[i] = a[i] + b[i];                  // the last block may overhang n
}

static void run(int n, bool cpu_only) {
    const size_t bytes = size_t(n) * sizeof(float);
    const double gb_moved = 3.0 * bytes / 1e9;  // read a, read b, write c

    std::vector<float> a(n), b(n), c_cpu(n), c_gpu(n);
    fill_random(a, -1.0f, 1.0f, 1);
    fill_random(b, -1.0f, 1.0f, 2);

    // ---- CPU -------------------------------------------------------------
    const double cpu_ms = median_ms([&] {
        CpuTimer t;
        t.start();
        vector_add_cpu(a.data(), b.data(), c_cpu.data(), n);
        return t.stop_ms();
    });
    print_row("vector_add", "cpu_sequential", n, cpu_ms, gb_moved / (cpu_ms / 1e3),
              "GB/s", 0.0, "PASS");
    if (cpu_only) return;

    // ---- GPU -------------------------------------------------------------
    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));  // memory on the GPU
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));

    const int threads = 256;                           // threads per block
    const int blocks = (n + threads - 1) / threads;    // enough blocks to cover n

    // (1) Kernel only: the inputs are already in GPU memory.
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
    const double kernel_ms = median_ms([&] {
        GpuTimer t;
        t.start();
        vector_add_gpu<<<blocks, threads>>>(d_a, d_b, d_c, n);
        CUDA_CHECK_LAUNCH();
        return t.stop_ms();
    });

    // (2) End to end: copy inputs over PCIe, compute, copy the result back.
    const double total_ms = median_ms([&] {
        GpuTimer t;
        t.start();
        CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes, cudaMemcpyHostToDevice));
        vector_add_gpu<<<blocks, threads>>>(d_a, d_b, d_c, n);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpy(c_gpu.data(), d_c, bytes, cudaMemcpyDeviceToHost));
        return t.stop_ms();
    });

    // Both sides perform the identical float addition, so results must match exactly.
    double err = 0.0;
    for (int i = 0; i < n; ++i)
        err = std::max(err, (double)std::fabs(c_gpu[i] - c_cpu[i]));
    const char* check = (err == 0.0) ? "PASS" : "FAIL";

    print_row("vector_add", "gpu_kernel", n, kernel_ms, gb_moved / (kernel_ms / 1e3),
              "GB/s", err, check);
    print_row("vector_add", "gpu_incl_transfers", n, total_ms,
              gb_moved / (total_ms / 1e3), "GB/s", err, check);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
}

int main(int argc, char** argv) {
    const bool cpu_only = has_flag(argc, argv, "--cpu-only");
    if (!cpu_only) print_device_info();
    const auto sizes = sizes_from_args(
        argc, argv, {1 << 10, 1 << 14, 1 << 18, 1 << 20, 1 << 22, 1 << 24, 1 << 26});
    print_csv_header();
    for (long long n : sizes) run(int(n), cpu_only);
    return 0;
}

In [ ]:
%%writefile matmul.cu
// matmul.cu - C = A x B for square n x n matrices of floats
//
// CPU:  triple loop on one core (i-k-j order, the cache-friendly one).
// GPU1: naive kernel, one thread per output element, reads straight from global memory.
// GPU2: tiled kernel, threads in a block share 16 x 16 tiles through fast shared memory.
// GPU3: cuBLAS, NVIDIA's tuned library, as the "what experts get" reference.
//
// Build: nvcc -O3 -std=c++17 -arch=native matmul.cu -o matmul -lcublas
// Run:   ./matmul               (default sizes)
//        ./matmul 1024 2048     (your own sizes)
#include "common.h"

#include <cublas_v2.h>

#define CUBLAS_CHECK(call)                                                   \
    do {                                                                     \
        cublasStatus_t st_ = (call);                                         \
        if (st_ != CUBLAS_STATUS_SUCCESS) {                                  \
            std::fprintf(stderr, "cuBLAS error %d at %s:%d\n", int(st_),     \
                         __FILE__, __LINE__);                                \
            std::exit(EXIT_FAILURE);                                         \
        }                                                                    \
    } while (0)

constexpr int TILE = 16;  // 16 x 16 = 256 threads per block

// ---- CPU: 2 n^3 floating-point operations, one after another ----------------
void matmul_cpu(const float* A, const float* B, float* C, int n) {
    std::fill(C, C + size_t(n) * n, 0.0f);
    for (int i = 0; i < n; ++i)
        for (int k = 0; k < n; ++k) {
            const float a = A[size_t(i) * n + k];
            for (int j = 0; j < n; ++j)  // walks rows of B and C contiguously
                C[size_t(i) * n + j] += a * B[size_t(k) * n + j];
        }
}

// ---- GPU 1: naive -----------------------------------------------------------
// Thread (row, col) computes one dot product. Every operand comes from global
// memory, and neighbouring threads re-read the same values many times.
__global__ void matmul_naive(const float* A, const float* B, float* C, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < n && col < n) {
        float sum = 0.0f;
        for (int k = 0; k < n; ++k) sum += A[row * n + k] * B[k * n + col];
        C[row * n + col] = sum;
    }
}

// ---- GPU 2: tiled with shared memory ----------------------------------------
// The block walks across A's row band and down B's column band one TILE x TILE
// tile at a time. Each thread loads ONE element of each tile into shared memory,
// the block waits (__syncthreads), then every thread reuses the whole tile.
// Global-memory traffic drops by a factor of TILE.
__global__ void matmul_tiled(const float* A, const float* B, float* C, int n) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    const int tx = threadIdx.x, ty = threadIdx.y;
    const int row = blockIdx.y * TILE + ty;
    const int col = blockIdx.x * TILE + tx;
    float sum = 0.0f;

    for (int t = 0; t < (n + TILE - 1) / TILE; ++t) {
        const int a_col = t * TILE + tx;  // column of A this thread loads
        const int b_row = t * TILE + ty;  // row of B this thread loads
        As[ty][tx] = (row < n && a_col < n) ? A[row * n + a_col] : 0.0f;
        Bs[ty][tx] = (b_row < n && col < n) ? B[b_row * n + col] : 0.0f;
        __syncthreads();  // the whole tile is loaded before anyone uses it

        for (int k = 0; k < TILE; ++k) sum += As[ty][k] * Bs[k][tx];
        __syncthreads();  // everyone is done with the tile before it is overwritten
    }
    if (row < n && col < n) C[row * n + col] = sum;
}

// Largest |gpu - cpu|, relative to the largest |cpu| value. Float sums done in a
// different order differ slightly, so we allow a small tolerance.
static double rel_error(const std::vector<float>& gpu, const std::vector<float>& cpu) {
    double max_diff = 0.0, max_ref = 0.0;
    for (size_t i = 0; i < cpu.size(); ++i) {
        max_diff = std::max(max_diff, (double)std::fabs(gpu[i] - cpu[i]));
        max_ref = std::max(max_ref, (double)std::fabs(cpu[i]));
    }
    return max_ref > 0.0 ? max_diff / max_ref : max_diff;
}

static void run(int n, bool cpu_only, cublasHandle_t handle) {
    const size_t elems = size_t(n) * n, bytes = elems * sizeof(float);
    const double gflop = 2.0 * double(n) * n * n / 1e9;
    const double tol = 1e-3;

    std::vector<float> A(elems), B(elems), C_cpu(elems), C_gpu(elems);
    fill_random(A, -1.0f, 1.0f, 3);
    fill_random(B, -1.0f, 1.0f, 4);

    // ---- CPU -------------------------------------------------------------
    const double cpu_ms = median_ms([&] {
        CpuTimer t;
        t.start();
        matmul_cpu(A.data(), B.data(), C_cpu.data(), n);
        return t.stop_ms();
    }, 3);
    print_row("matmul", "cpu_sequential", n, cpu_ms, gflop / (cpu_ms / 1e3), "GFLOP/s",
              0.0, "PASS");
    if (cpu_only) return;

    // ---- GPU -------------------------------------------------------------
    float *dA = nullptr, *dB = nullptr, *dC = nullptr;
    CUDA_CHECK(cudaMalloc(&dA, bytes));
    CUDA_CHECK(cudaMalloc(&dB, bytes));
    CUDA_CHECK(cudaMalloc(&dC, bytes));
    CUDA_CHECK(cudaMemcpy(dA, A.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, B.data(), bytes, cudaMemcpyHostToDevice));

    const dim3 block(TILE, TILE);
    const dim3 grid((n + TILE - 1) / TILE, (n + TILE - 1) / TILE);

    auto report = [&](const char* variant, double ms) {
        CUDA_CHECK(cudaMemcpy(C_gpu.data(), dC, bytes, cudaMemcpyDeviceToHost));
        const double err = rel_error(C_gpu, C_cpu);
        print_row("matmul", variant, n, ms, gflop / (ms / 1e3), "GFLOP/s", err,
                  err <= tol ? "PASS" : "FAIL");
    };

    // GPU 1: naive
    double ms = median_ms([&] {
        GpuTimer t;
        t.start();
        matmul_naive<<<grid, block>>>(dA, dB, dC, n);
        CUDA_CHECK_LAUNCH();
        return t.stop_ms();
    });
    report("gpu_naive_kernel", ms);

    // GPU 2: tiled
    ms = median_ms([&] {
        GpuTimer t;
        t.start();
        matmul_tiled<<<grid, block>>>(dA, dB, dC, n);
        CUDA_CHECK_LAUNCH();
        return t.stop_ms();
    });
    report("gpu_tiled_kernel", ms);

    // GPU 3: cuBLAS. cuBLAS assumes column-major storage; computing B x A in
    // column-major order gives exactly A x B in our row-major layout.
    const float alpha = 1.0f, beta = 0.0f;
    ms = median_ms([&] {
        GpuTimer t;
        t.start();
        CUBLAS_CHECK(cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, n, n, n, &alpha,
                                 dB, n, dA, n, &beta, dC, n));
        return t.stop_ms();
    });
    report("gpu_cublas", ms);

    // Tiled kernel again, this time counting the copies in and out.
    ms = median_ms([&] {
        GpuTimer t;
        t.start();
        CUDA_CHECK(cudaMemcpy(dA, A.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(dB, B.data(), bytes, cudaMemcpyHostToDevice));
        matmul_tiled<<<grid, block>>>(dA, dB, dC, n);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpy(C_gpu.data(), dC, bytes, cudaMemcpyDeviceToHost));
        return t.stop_ms();
    });
    report("gpu_tiled_incl_transfers", ms);

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
}

int main(int argc, char** argv) {
    const bool cpu_only = has_flag(argc, argv, "--cpu-only");
    cublasHandle_t handle = nullptr;
    if (!cpu_only) {
        print_device_info();
        CUBLAS_CHECK(cublasCreate(&handle));
    }
    const auto sizes = sizes_from_args(argc, argv, {128, 256, 512, 1024, 2048});
    print_csv_header();
    for (long long n : sizes) run(int(n), cpu_only, handle);
    if (handle) cublasDestroy(handle);
    return 0;
}

In [ ]:
%%writefile image_blur.cu
// image_blur.cu - box blur of a grayscale image
//
// Every output pixel is the average of the (2r+1) x (2r+1) square of input pixels
// around it (r = radius). Edges are handled by clamping to the nearest pixel.
//
// CPU: two nested loops visit the pixels one at a time.
// GPU: a 2-D grid of 16 x 16 thread blocks, one thread per pixel.
// Both call the SAME blur_pixel() function: only the scheduling changes.
//
// Build: nvcc -O3 -std=c++17 -arch=native image_blur.cu -o image_blur
// Run:   ./image_blur                               (synthetic images, default sizes)
//        ./image_blur --radius 8 1024 4096          (your own radius and sizes)
//        ./image_blur --in photo.pgm --out blurred  (a real image; writes
//                                                    blurred_cpu.pgm and blurred_gpu.pgm)
#include "common.h"

#include <fstream>

struct Image {
    int w = 0, h = 0;
    std::vector<unsigned char> px;  // row-major, one byte per pixel
};

// ---- PGM (portable graymap) input/output -------------------------------------
static bool read_pgm(const std::string& path, Image& img) {
    std::ifstream f(path, std::ios::binary);
    if (!f) return false;
    std::string magic;
    f >> magic;
    if (magic != "P5") return false;
    auto next_int = [&f]() {  // read an integer, skipping '#' comment lines
        int value = 0;
        while (f >> std::ws && f.peek() == '#') f.ignore(1 << 20, '\n');
        f >> value;
        return value;
    };
    img.w = next_int();
    img.h = next_int();
    const int maxval = next_int();
    f.get();  // the single whitespace byte before the pixel data
    if (img.w <= 0 || img.h <= 0 || maxval != 255) return false;
    img.px.resize(size_t(img.w) * img.h);
    f.read(reinterpret_cast<char*>(img.px.data()), img.px.size());
    return bool(f);
}

static bool write_pgm(const std::string& path, const Image& img) {
    std::ofstream f(path, std::ios::binary);
    f << "P5\n" << img.w << " " << img.h << "\n255\n";
    f.write(reinterpret_cast<const char*>(img.px.data()), img.px.size());
    return bool(f);
}

// A synthetic test image: smooth gradients, rings and some pixel noise.
static Image make_test_image(int w, int h) {
    Image img;
    img.w = w;
    img.h = h;
    img.px.resize(size_t(w) * h);
    std::mt19937 rng(7);
    std::uniform_int_distribution<int> noise(-40, 40);
    for (int y = 0; y < h; ++y)
        for (int x = 0; x < w; ++x) {
            const double dx = x - w / 2.0, dy = y - h / 2.0;
            const double ring = 0.5 + 0.5 * std::cos(std::sqrt(dx * dx + dy * dy) / (w / 40.0));
            int v = int(80.0 * x / w + 60.0 * y / h + 110.0 * ring) + noise(rng);
            img.px[size_t(y) * w + x] = (unsigned char)std::min(255, std::max(0, v));
        }
    return img;
}

// ---- the per-pixel work, shared by CPU and GPU ----------------------------------
__host__ __device__ inline int clamp_int(int v, int lo, int hi) {
    return v < lo ? lo : (v > hi ? hi : v);
}

__host__ __device__ inline unsigned char blur_pixel(const unsigned char* in, int w, int h,
                                                    int x, int y, int r) {
    int sum = 0;
    for (int dy = -r; dy <= r; ++dy) {
        const int yy = clamp_int(y + dy, 0, h - 1);
        for (int dx = -r; dx <= r; ++dx) {
            const int xx = clamp_int(x + dx, 0, w - 1);
            sum += in[yy * w + xx];
        }
    }
    const int area = (2 * r + 1) * (2 * r + 1);
    return (unsigned char)((sum + area / 2) / area);  // rounded average
}

// CPU: visit every pixel in turn.
void box_blur_cpu(const unsigned char* in, unsigned char* out, int w, int h, int r) {
    for (int y = 0; y < h; ++y)
        for (int x = 0; x < w; ++x) out[y * w + x] = blur_pixel(in, w, h, x, y, r);
}

// GPU: each thread finds its own (x, y) and computes that one pixel.
__global__ void box_blur_gpu(const unsigned char* in, unsigned char* out, int w, int h,
                             int r) {
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x < w && y < h) out[y * w + x] = blur_pixel(in, w, h, x, y, r);
}

static void run(const Image& input, int r, bool cpu_only, const char* out_prefix) {
    const int w = input.w, h = input.h;
    const size_t bytes = size_t(w) * h;
    const long long pixels = (long long)w * h;
    const double mpix = pixels / 1e6;

    Image out_cpu = input, out_gpu = input;

    // ---- CPU -------------------------------------------------------------
    const double cpu_ms = median_ms([&] {
        CpuTimer t;
        t.start();
        box_blur_cpu(input.px.data(), out_cpu.px.data(), w, h, r);
        return t.stop_ms();
    }, 3);
    print_row("image_blur", "cpu_sequential", pixels, cpu_ms, mpix / (cpu_ms / 1e3),
              "Mpixel/s", 0.0, "PASS");
    if (out_prefix) write_pgm(std::string(out_prefix) + "_cpu.pgm", out_cpu);
    if (cpu_only) return;

    // ---- GPU -------------------------------------------------------------
    unsigned char *d_in = nullptr, *d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_in, bytes));
    CUDA_CHECK(cudaMalloc(&d_out, bytes));
    CUDA_CHECK(cudaMemcpy(d_in, input.px.data(), bytes, cudaMemcpyHostToDevice));

    const dim3 block(16, 16);
    const dim3 grid((w + block.x - 1) / block.x, (h + block.y - 1) / block.y);

    const double kernel_ms = median_ms([&] {
        GpuTimer t;
        t.start();
        box_blur_gpu<<<grid, block>>>(d_in, d_out, w, h, r);
        CUDA_CHECK_LAUNCH();
        return t.stop_ms();
    });

    const double total_ms = median_ms([&] {
        GpuTimer t;
        t.start();
        CUDA_CHECK(cudaMemcpy(d_in, input.px.data(), bytes, cudaMemcpyHostToDevice));
        box_blur_gpu<<<grid, block>>>(d_in, d_out, w, h, r);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpy(out_gpu.px.data(), d_out, bytes, cudaMemcpyDeviceToHost));
        return t.stop_ms();
    });

    // Integer arithmetic: the two images must be identical, pixel for pixel.
    int max_diff = 0;
    for (size_t i = 0; i < bytes; ++i)
        max_diff = std::max(max_diff, std::abs(int(out_gpu.px[i]) - int(out_cpu.px[i])));
    const char* check = max_diff == 0 ? "PASS" : "FAIL";

    print_row("image_blur", "gpu_kernel", pixels, kernel_ms, mpix / (kernel_ms / 1e3),
              "Mpixel/s", max_diff, check);
    print_row("image_blur", "gpu_incl_transfers", pixels, total_ms,
              mpix / (total_ms / 1e3), "Mpixel/s", max_diff, check);
    if (out_prefix) write_pgm(std::string(out_prefix) + "_gpu.pgm", out_gpu);

    CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_out));
}

int main(int argc, char** argv) {
    const bool cpu_only = has_flag(argc, argv, "--cpu-only");
    const int r = std::atoi(option_value(argc, argv, "--radius", "4"));
    const char* in_path = option_value(argc, argv, "--in", nullptr);
    const char* out_prefix = option_value(argc, argv, "--out", nullptr);

    if (!cpu_only) print_device_info();
    std::printf("# radius %d: each pixel averages %d neighbours\n", r,
                (2 * r + 1) * (2 * r + 1));
    print_csv_header();

    if (in_path) {  // one real image
        Image img;
        if (!read_pgm(in_path, img)) {
            std::fprintf(stderr, "Could not read %s (expected binary PGM, P5)\n", in_path);
            return EXIT_FAILURE;
        }
        run(img, r, cpu_only, out_prefix);
        return 0;
    }
    const auto sides = sizes_from_args(argc, argv, {256, 512, 1024, 2048, 4096});
    for (long long s : sides) run(make_test_image(int(s), int(s)), r, cpu_only, out_prefix);
    return 0;
}

In [ ]:
%%writefile reduction.cu
// reduction.cu - sum of n floats
//
// Summation is the classic example of a problem that is NOT naturally parallel:
// every addition depends on the running total. This file shows four ways:
//
// CPU:              one running total, n additions in a row.
// GPU racy:         every thread does `*total += x[i]`. Fast and WRONG: a race condition.
// GPU atomic:       every thread does atomicAdd(total, x[i]). Correct, but the additions
//                   to one memory address are serialized, so it is slow.
// GPU shared tree:  each block sums its share in shared memory as a tree of pairwise
//                   additions; one atomicAdd per block. Correct and fast.
//
// The reference answer is computed in double precision (not timed).
//
// Build: nvcc -O3 -std=c++17 -arch=native reduction.cu -o reduction
// Run:   ./reduction              (default sizes)
//        ./reduction 100000000    (your own size)
#include "common.h"

constexpr int BLOCK = 256;       // threads per block (a power of two for the tree)
constexpr int MAX_BLOCKS = 1024; // the shared-tree kernel loops if n needs more

// ---- CPU ---------------------------------------------------------------------
float sum_cpu(const float* x, int n) {
    float total = 0.0f;
    for (int i = 0; i < n; ++i) total += x[i];
    return total;
}

double sum_reference(const float* x, int n) {
    double total = 0.0;
    for (int i = 0; i < n; ++i) total += x[i];
    return total;
}

// ---- GPU: racy (wrong on purpose) ------------------------------------------------
// Thousands of threads read the same old value of *total, add to it, and write it
// back, overwriting each other's updates. Most additions are lost.
__global__ void sum_racy(const float* x, float* total, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) *total += x[i];
}

// ---- GPU: one atomic per element -----------------------------------------------
// atomicAdd makes each read-add-write indivisible, so no update is lost. But all n
// threads queue for the same address.
__global__ void sum_atomic(const float* x, float* total, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) atomicAdd(total, x[i]);
}

// ---- GPU: shared-memory tree reduction -----------------------------------------
//  1. Each thread adds up a strided slice of x in a private register (grid-stride loop).
//  2. The block's 256 partial sums go into shared memory.
//  3. Pairwise tree: 256 -> 128 -> 64 -> ... -> 1, halving the active threads each step.
//  4. Thread 0 adds the block's total to the global result: one atomic per block.
__global__ void sum_shared_tree(const float* x, float* total, int n) {
    __shared__ float partial[BLOCK];

    float local = 0.0f;
    for (int i = blockIdx.x * BLOCK + threadIdx.x; i < n; i += BLOCK * gridDim.x)
        local += x[i];
    partial[threadIdx.x] = local;
    __syncthreads();

    for (int stride = BLOCK / 2; stride > 0; stride /= 2) {
        if (threadIdx.x < stride) partial[threadIdx.x] += partial[threadIdx.x + stride];
        __syncthreads();  // finish this level of the tree before starting the next
    }
    if (threadIdx.x == 0) atomicAdd(total, partial[0]);
}

static void run(int n, bool cpu_only) {
    const size_t bytes = size_t(n) * sizeof(float);
    const double gb = bytes / 1e9;
    const double tol = 1e-3;  // allowed relative error

    std::vector<float> x(n);
    fill_random(x, 0.0f, 1.0f, 5);
    const double ref = sum_reference(x.data(), n);
    auto rel = [ref](double v) { return std::fabs(v - ref) / std::fabs(ref); };

    // ---- CPU -------------------------------------------------------------
    float cpu_result = 0.0f;
    const double cpu_ms = median_ms([&] {
        CpuTimer t;
        t.start();
        cpu_result = sum_cpu(x.data(), n);
        return t.stop_ms();
    });
    const double cpu_err = rel(cpu_result);
    print_row("reduction", "cpu_sequential", n, cpu_ms, gb / (cpu_ms / 1e3), "GB/s",
              cpu_err, cpu_err <= tol ? "PASS" : "INACCURATE");
    if (cpu_only) return;

    // ---- GPU -------------------------------------------------------------
    float *d_x = nullptr, *d_total = nullptr;
    CUDA_CHECK(cudaMalloc(&d_x, bytes));
    CUDA_CHECK(cudaMalloc(&d_total, sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, x.data(), bytes, cudaMemcpyHostToDevice));

    const int one_per_thread_blocks = (n + BLOCK - 1) / BLOCK;
    const int tree_blocks = std::min(one_per_thread_blocks, MAX_BLOCKS);

    // Times one variant (reset total + kernel) and reports its result.
    auto time_variant = [&](const char* variant, auto launch, const char* bad_label) {
        const double ms = median_ms([&] {
            GpuTimer t;
            t.start();
            CUDA_CHECK(cudaMemsetAsync(d_total, 0, sizeof(float)));
            launch();
            CUDA_CHECK_LAUNCH();
            return t.stop_ms();
        });
        float result = 0.0f;
        CUDA_CHECK(cudaMemcpy(&result, d_total, sizeof(float), cudaMemcpyDeviceToHost));
        const double err = rel(result);
        print_row("reduction", variant, n, ms, gb / (ms / 1e3), "GB/s", err,
                  err <= tol ? "PASS" : bad_label);
    };

    time_variant("gpu_racy_kernel", [&] {
        sum_racy<<<one_per_thread_blocks, BLOCK>>>(d_x, d_total, n);
    }, "WRONG_RACE");
    time_variant("gpu_atomic_kernel", [&] {
        sum_atomic<<<one_per_thread_blocks, BLOCK>>>(d_x, d_total, n);
    }, "INACCURATE");
    time_variant("gpu_shared_tree_kernel", [&] {
        sum_shared_tree<<<tree_blocks, BLOCK>>>(d_x, d_total, n);
    }, "INACCURATE");

    // Shared tree again, counting the copy of x to the GPU and the result back.
    float result = 0.0f;
    const double total_ms = median_ms([&] {
        GpuTimer t;
        t.start();
        CUDA_CHECK(cudaMemcpy(d_x, x.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemsetAsync(d_total, 0, sizeof(float)));
        sum_shared_tree<<<tree_blocks, BLOCK>>>(d_x, d_total, n);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpy(&result, d_total, sizeof(float), cudaMemcpyDeviceToHost));
        return t.stop_ms();
    });
    const double err = rel(result);
    print_row("reduction", "gpu_shared_tree_incl_transfers", n, total_ms,
              gb / (total_ms / 1e3), "GB/s", err, err <= tol ? "PASS" : "INACCURATE");

    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_total));
}

int main(int argc, char** argv) {
    const bool cpu_only = has_flag(argc, argv, "--cpu-only");
    if (!cpu_only) print_device_info();
    const auto sizes =
        sizes_from_args(argc, argv, {1 << 10, 1 << 16, 1 << 20, 1 << 22, 1 << 24, 1 << 26});
    print_csv_header();
    for (long long n : sizes) run(int(n), cpu_only);
    return 0;
}

## 3. Compile

`-arch` tells `nvcc` which GPU generation to build for (`sm_75` = T4, `sm_80` = A100). The host (CPU) code gets `-O3 -march=native`, so the CPU baseline is a fair, optimized single-core loop.

In [ ]:
FLAGS = f"-O3 -std=c++17 -Xcompiler -O3,-march=native -arch={ARCH}"
!nvcc {FLAGS} vector_add.cu -o vector_add
!nvcc {FLAGS} matmul.cu -o matmul -lcublas
!nvcc {FLAGS} image_blur.cu -o image_blur
!nvcc {FLAGS} reduction.cu -o reduction

PROGRAMS = ["vector_add", "matmul", "image_blur", "reduction"]
missing = [p for p in PROGRAMS if not os.path.exists(p)]
if missing:
    raise SystemExit(f"Build failed for: {missing} (see the errors above)")
print("Built:", ", ".join(PROGRAMS))

## 4. Run the benchmarks

Each program prints one CSV row per measurement: `time_ms` is the median of 5 runs after a warm-up (slow CPU runs are timed once). `check` says whether the GPU answer matched the CPU answer.

The summation benchmark contains one variant that is **wrong on purpose** (`gpu_racy_kernel`, check = `WRONG_RACE`), and the CPU's own float sum becomes `INACCURATE` at 67 million elements. The tutorial explains both.

In [ ]:
for prog in PROGRAMS:
    print(f"===== {prog}")
    run = subprocess.run([f"./{prog}"], capture_output=True, text=True)
    print(run.stdout)
    if run.returncode != 0:
        print(run.stderr)
        raise SystemExit(f"{prog} failed")
    with open(f"results/{prog}.csv", "w") as f:
        f.write(run.stdout)

## 5. See the blur on a real photo

The blur program reads a grayscale PGM image, blurs it on both the CPU and the GPU, and writes both results. The two images must be identical, pixel for pixel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from skimage import data
    photo = data.camera()                      # 512 x 512 grayscale test photo
except Exception:
    yy, xx = np.mgrid[0:512, 0:512]            # fallback: a synthetic pattern
    photo = (((xx // 32 + yy // 32) % 2) * 160 + 40).astype(np.uint8)

def write_pgm(path, img):
    h, w = img.shape
    with open(path, "wb") as f:
        f.write(f"P5\n{w} {h}\n255\n".encode() + img.astype(np.uint8).tobytes())

def read_pgm(path):
    magic, size, maxval, pixels = open(path, "rb").read().split(b"\n", 3)
    w, h = map(int, size.split())
    return np.frombuffer(pixels, dtype=np.uint8).reshape(h, w)

RADIUS = 6
write_pgm("photo.pgm", photo)
demo = subprocess.run(["./image_blur", "--radius", str(RADIUS), "--in", "photo.pgm", "--out", "photo_blur"],
                      capture_output=True, text=True)
print(demo.stdout or demo.stderr)

cpu_img, gpu_img = read_pgm("photo_blur_cpu.pgm"), read_pgm("photo_blur_gpu.pgm")
same = np.array_equal(cpu_img, gpu_img)
side = 2 * RADIUS + 1
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
for ax, img, title in zip(axes, [photo, cpu_img, gpu_img], ["Original", "Blurred on the CPU", "Blurred on the GPU"]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
fig.suptitle(f"Box blur, {side} x {side} average: the CPU and GPU images are "
             f"{'identical' if same else 'DIFFERENT (check the code!)'}", fontsize=12)
fig.savefig("results/blur_demo.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Tables: time, throughput and speedup

`speedup_vs_cpu` = CPU time ÷ variant time for the same problem size. Above 1 means faster than one CPU core.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160)

df = pd.concat([pd.read_csv(f"results/{p}.csv", comment="#") for p in PROGRAMS], ignore_index=True)
cpu = (df[df.variant == "cpu_sequential"][["example", "size", "time_ms"]]
       .rename(columns={"time_ms": "cpu_ms"}))
df = df.merge(cpu, on=["example", "size"])
df["speedup_vs_cpu"] = df.cpu_ms / df.time_ms
df.to_csv("results/all_results.csv", index=False)

LABEL = {
    "cpu_sequential": "CPU, 1 core",
    "gpu_kernel": "GPU kernel only",
    "gpu_incl_transfers": "GPU incl. transfers",
    "gpu_naive_kernel": "GPU naive kernel",
    "gpu_tiled_kernel": "GPU tiled kernel",
    "gpu_cublas": "GPU cuBLAS",
    "gpu_tiled_incl_transfers": "GPU tiled incl. transfers",
    "gpu_racy_kernel": "GPU racy (wrong)",
    "gpu_atomic_kernel": "GPU atomic per element",
    "gpu_shared_tree_kernel": "GPU shared-memory tree",
    "gpu_shared_tree_incl_transfers": "GPU tree incl. transfers",
}
TITLE = {"vector_add": "Vector addition", "matmul": "Matrix multiplication",
         "image_blur": "Image blur", "reduction": "Summation"}
SIZE_NOTE = {"vector_add": "elements", "matmul": "n (n x n matrices)",
             "image_blur": "pixels", "reduction": "elements"}

for ex in PROGRAMS:
    part = df[df.example == ex]
    table = part.pivot(index="size", columns="variant", values="speedup_vs_cpu")
    table = table[[v for v in LABEL if v in table.columns and v != "cpu_sequential"]]
    table.columns = [LABEL[c] for c in table.columns]
    table.index.name = SIZE_NOTE[ex]
    print(f"\n{TITLE[ex]}: speedup vs one CPU core (x)")
    print(table.round(2).to_string())

## 7. Charts

In [ ]:
# One fixed colour per role, the same in every panel. CPU stays grey: it is the baseline.
ROLE = {
    "cpu_sequential": "cpu",
    "gpu_kernel": "best", "gpu_tiled_kernel": "best", "gpu_shared_tree_kernel": "best",
    "gpu_incl_transfers": "transfers", "gpu_tiled_incl_transfers": "transfers",
    "gpu_shared_tree_incl_transfers": "transfers",
    "gpu_naive_kernel": "alt", "gpu_atomic_kernel": "alt",
    "gpu_cublas": "library",
}
COLOR = {"cpu": "#898781", "best": "#2a78d6", "transfers": "#eb6834", "alt": "#1baf7a", "library": "#eda100"}
MARKER = {"cpu": "o", "best": "s", "transfers": "^", "alt": "D", "library": "v"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"

plt.rcParams.update({"axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10})

valid = df[df.check != "WRONG_RACE"]          # a wrong answer's speed is meaningless

# ---- Chart 1: time vs problem size, one panel per example ----
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
for ax, ex in zip(axes.flat, PROGRAMS):
    part = valid[valid.example == ex]
    for v in [v for v in LABEL if v in set(part.variant)]:
        s = part[part.variant == v].sort_values("size")
        role = ROLE[v]
        ax.plot(s["size"], s.time_ms, color=COLOR[role], marker=MARKER[role], markersize=6,
                linewidth=2, label=LABEL[v], zorder=3 if role != "cpu" else 2)
    ax.set_xscale("log"); ax.set_yscale("log")
    if ex == "matmul":                          # few sizes: label each one
        sizes = sorted(part["size"].unique())
        ax.set_xticks(sizes); ax.set_xticklabels([str(n) for n in sizes]); ax.minorticks_off()
    ax.grid(True, which="major", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.set_title(TITLE[ex], loc="left", color=INK, fontsize=12)
    ax.set_xlabel(f"problem size ({SIZE_NOTE[ex]}, log scale)")
    ax.set_ylabel("time per run (ms, log scale)")
    ax.legend(frameon=False, fontsize=8.5)
fig.suptitle(f"Time per run vs problem size on {gpu_name} (lower is faster)", x=0.01, ha="left",
             fontsize=13, color=INK)
fig.tight_layout()
fig.savefig("results/time_vs_size.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Chart 2: speedup at the largest size of each example ----
largest = valid[valid["size"] == valid.groupby("example")["size"].transform("max")]
bars = largest[largest.variant != "cpu_sequential"].copy()
bars["order"] = bars.example.map({e: i for i, e in enumerate(PROGRAMS)})
bars = bars.sort_values(["order", "speedup_vs_cpu"], ascending=[True, False])
best = bars[bars.variant.map(ROLE) == "best"]

fig, ax = plt.subplots(figsize=(10, 0.42 * len(bars) + 1.6))
y = np.arange(len(bars))[::-1]
ax.barh(y, bars.speedup_vs_cpu, color=[COLOR[ROLE[v]] for v in bars.variant], height=0.68,
        edgecolor="#fcfcfb", linewidth=2)
ax.set_xscale("log")
ax.axvline(1, color=MUTED, linewidth=1.2, linestyle="--")
ax.text(1, y.max() + 0.75, " same speed as 1 CPU core", color=MUTED, fontsize=9, va="center")
ax.set_yticks(y)
ax.set_yticklabels([f"{TITLE[e]}: {LABEL[v]}" for e, v in zip(bars.example, bars.variant)], fontsize=9)
for yi, val in zip(y, bars.speedup_vs_cpu):
    ax.text(val * 1.08, yi, f"{val:,.1f}x" if val < 100 else f"{val:,.0f}x", va="center",
            fontsize=9, color=INK)
ax.set_xlabel("speedup vs one CPU core (log scale)")
ax.grid(True, axis="x", which="major", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
ax.set_xlim(right=bars.speedup_vs_cpu.max() * 3)
fmt = lambda v: f"{v:,.1f}x" if v < 10 else f"{v:,.0f}x"
ax.set_title(f"At the largest sizes, the best GPU kernels run at {fmt(best.speedup_vs_cpu.min())} to "
             f"{fmt(best.speedup_vs_cpu.max())} the speed of one CPU core", loc="left", color=INK, fontsize=12)
fig.tight_layout()
fig.savefig("results/speedup_largest.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary to copy into the tutorial

The table below is Markdown, ready to paste. It also reports where the GPU starts paying off once the copies to and from the GPU are counted.

In [ ]:
rows = ["| Example | Size | CPU (ms) | Best GPU kernel (ms) | Speedup, kernel | GPU incl. transfers (ms) | Speedup, incl. transfers |",
        "| --- | --- | --- | --- | --- | --- | --- |"]
for ex in PROGRAMS:
    part = valid[valid.example == ex]
    n = part["size"].max()
    at_n = part[part["size"] == n]
    k = at_n[at_n.variant.map(ROLE) == "best"].iloc[0]
    t = at_n[at_n.variant.map(ROLE) == "transfers"].iloc[0]
    rows.append(f"| {TITLE[ex]} | {n:,} | {k.cpu_ms:,.2f} | {k.time_ms:,.3f} | {k.speedup_vs_cpu:,.1f}x "
                f"| {t.time_ms:,.2f} | {t.speedup_vs_cpu:,.1f}x |")
summary_md = "\n".join(rows)
print(summary_md)
open("results/summary.md", "w").write(summary_md + "\n")

print("\nSmallest size where the GPU wins even after counting transfers:")
for ex in PROGRAMS:
    part = valid[(valid.example == ex) & (valid.variant.map(ROLE) == "transfers")].sort_values("size")
    wins = part[part.speedup_vs_cpu > 1]
    print(f"  {TITLE[ex]:<22} " + (f"{wins['size'].iloc[0]:,} {SIZE_NOTE[ex]}" if len(wins) else "never in this range"))

In [ ]:
# Bundle every CSV and chart for download.
shutil.make_archive("cuda_results", "zip", "results")
print("Saved cuda_examples/cuda_results.zip")
try:
    from google.colab import files     # only exists on Colab
    files.download("cuda_results.zip")
except ImportError:
    print("Not on Colab: download cuda_results.zip from the file browser (Kaggle: the Output panel).")

## What to look for

- **Vector addition:** the kernel alone is fast, but copying the arrays over PCIe takes longer than the CPU loop. Compare `GPU kernel only` with `GPU incl. transfers`.
- **Matrix multiplication:** work grows as n³ while data grows as n², so the GPU wins even with transfers. Tiling beats the naive kernel; cuBLAS beats both.
- **Image blur:** every pixel is independent, and the CPU and GPU images are identical.
- **Summation:** the racy kernel is wrong, the atomic kernel is slow, and the tree is both fast and more accurate than the CPU's running float sum.